## 1.2 WDI Data Cleaning

Filter, reshape, interpolate, and validate World Development Indicators for Malaysia (MYS) and the Philippines (PHL) across 12 climate-risk-related series from 1990–2023.

In [1]:
import pandas as pd
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────────────────
RAW_PATH = Path("../data/raw/WB_WDI_WIDEF.csv")
OUT_DIR  = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COUNTRIES = ["MYS", "PHL"]
YEAR_RANGE = list(range(1990, 2024))          # 1990–2023 inclusive
YEAR_COLS  = [str(y) for y in YEAR_RANGE]

# Mapping: desired output series_code  →  INDICATOR value in the CSV
# The bulk CSV prefixes every code with WB_WDI_ and uses underscores.
# Two original codes were replaced per user instruction:
#   EN.ATM.GHGT.KT.CE  →  WB_WDI_EN_GHG_ALL_MT_CE_AR5
#   EN.ATM.CO2E.PC     →  WB_WDI_EN_GHG_CO2_PC_CE_AR5
SERIES_MAP = {
    "AG.LND.PRCP.MM":              "WB_WDI_AG_LND_PRCP_MM",
    "WB_WDI_EN_GHG_ALL_MT_CE_AR5": "WB_WDI_EN_GHG_ALL_MT_CE_AR5",
    "WB_WDI_EN_GHG_CO2_PC_CE_AR5": "WB_WDI_EN_GHG_CO2_PC_CE_AR5",
    "EN_GHG_CO2_RT_GDP_KD":        "WB_WDI_EN_GHG_CO2_RT_GDP_KD",
    "SP.URB.TOTL.IN.ZS":           "WB_WDI_SP_URB_TOTL_IN_ZS",
    "EN.CLC.MDAT.ZS":              "WB_WDI_EN_CLC_MDAT_ZS",
    "AG.LND.FRST.ZS":              "WB_WDI_AG_LND_FRST_ZS",
    "EG.FEC.RNEW.ZS":              "WB_WDI_EG_FEC_RNEW_ZS",
    "EG.USE.PCAP.KG.OE":           "WB_WDI_EG_USE_PCAP_KG_OE",
    "EG.USE.COMM.FO.ZS":           "WB_WDI_EG_USE_COMM_FO_ZS",
    "NY.GDP.PCAP.CD":              "WB_WDI_NY_GDP_PCAP_CD",
    "NV.IND.MANF.ZS":              "WB_WDI_NV_IND_MANF_ZS",
}
INDICATOR_TO_SERIES = {v: k for k, v in SERIES_MAP.items()}
TARGET_INDICATORS  = list(SERIES_MAP.values())
SERIES_CODES       = list(SERIES_MAP.keys())        # 12 output codes

# ── Step 1: Filter to target countries & indicators ───────────────────────
use_cols = ["REF_AREA", "INDICATOR"] + YEAR_COLS
df_raw = pd.read_csv(RAW_PATH, usecols=use_cols, dtype=str)

mask = (
    df_raw["REF_AREA"].isin(TARGET_COUNTRIES)
    & df_raw["INDICATOR"].isin(TARGET_INDICATORS)
)
df_filt = df_raw.loc[mask].copy()
df_filt["series_code"]  = df_filt["INDICATOR"].map(INDICATOR_TO_SERIES)
df_filt["country_code"] = df_filt["REF_AREA"]
df_filt.drop(columns=["REF_AREA", "INDICATOR"], inplace=True)

# ── Step 2: Melt to long panel format ─────────────────────────────────────
df_long = df_filt.melt(
    id_vars=["country_code", "series_code"],
    value_vars=YEAR_COLS,
    var_name="year",
    value_name="value",
)
df_long["year"]  = df_long["year"].astype(int)
df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")

# Ensure full panel: every country × indicator × year exists
full_idx = pd.MultiIndex.from_product(
    [TARGET_COUNTRIES, SERIES_CODES, YEAR_RANGE],
    names=["country_code", "series_code", "year"],
)
df_long = (
    df_long.set_index(["country_code", "series_code", "year"])
    .reindex(full_idx)
    .reset_index()
)
df_long = df_long[["country_code", "year", "series_code", "value"]]

# ── Step 3: Missing-data handling with logged actions ─────────────────────
log_rows = []

df_long = df_long.sort_values(
    ["country_code", "series_code", "year"]
).reset_index(drop=True)

for (cc, sc), grp in df_long.groupby(["country_code", "series_code"]):
    grp = grp.sort_values("year")
    idx = grp.index
    vals = grp["value"].values.copy()
    years = grp["year"].values
    n = len(vals)
    is_null = pd.isna(vals)

    # --- identify contiguous NaN gaps ---
    protect = set()                       # positions to keep as NaN
    i = 0
    while i < n:
        if is_null[i]:
            j = i
            while j < n and is_null[j]:
                j += 1
            gap_len = j - i
            has_left  = (i > 0)
            has_right = (j < n)

            if gap_len <= 2 and has_left and has_right:
                action = "interpolated"
                note   = f"linear interpolation, {gap_len}-year gap"
            elif not has_left or not has_right:
                action = "excluded"
                note   = "edge gap, no surrounding value for interpolation"
                protect.update(range(i, j))
            else:
                action = "excluded"
                note   = "gap >= 3 years, excluded from analysis"
                protect.update(range(i, j))

            log_rows.append({
                "country_code": cc,
                "series_code":  sc,
                "year_start":   int(years[i]),
                "year_end":     int(years[j - 1]),
                "gap_length":   gap_len,
                "action":       action,
                "note":         note,
            })
            i = j
        else:
            i += 1

    # --- interpolate (only interior gaps ≤ 2) ---
    s = grp["value"].copy()
    s = s.interpolate(method="linear", limit=2, limit_direction="both")

    # Re-apply NaN for protected positions (gaps ≥ 3 or edge)
    prot_iloc = sorted(protect)
    if prot_iloc:
        s.iloc[prot_iloc] = float("nan")

    df_long.loc[idx, "value"] = s.values

# Build missing-data log
df_log = pd.DataFrame(log_rows, columns=[
    "country_code", "series_code", "year_start",
    "year_end", "gap_length", "action", "note",
])
df_log.to_csv(OUT_DIR / "missing_data_log.csv", index=False)

# ── Step 4: Pivot to wide panel & save ────────────────────────────────────
df_wide = df_long.pivot_table(
    index=["country_code", "year"],
    columns="series_code",
    values="value",
    aggfunc="first",
).reset_index()
df_wide.columns.name = None
df_wide = df_wide.sort_values(["country_code", "year"]).reset_index(drop=True)

# Ensure column order: country_code, year, then the 12 series codes
df_wide = df_wide[["country_code", "year"] + SERIES_CODES]
df_wide.to_csv(OUT_DIR / "cleaned_wdi.csv", index=False)

# ── Step 5: Validation checks ─────────────────────────────────────────────
out = pd.read_csv(OUT_DIR / "cleaned_wdi.csv")

# 1. Exactly 68 rows
assert out.shape[0] == 68, f"Expected 68 rows, got {out.shape[0]}"

# 2. Exactly 14 columns
assert out.shape[1] == 14, f"Expected 14 columns, got {out.shape[1]}"

# 3. No indicator column is 100 % NaN for either country
for cc in TARGET_COUNTRIES:
    sub = out[out["country_code"] == cc]
    for sc in SERIES_CODES:
        assert not sub[sc].isna().all(), (
            f"{sc} is 100% NaN for {cc}"
        )

# 4. Log has ≥ 0 rows (always true, but assert explicitly)
log_check = pd.read_csv(OUT_DIR / "missing_data_log.csv")
assert len(log_check) >= 0, "Log has negative rows?!"

# 5. Missingness summary
print("=" * 60)
print("VALIDATION PASSED — all 4 assertions OK")
print("=" * 60)
print(f"Output rows : {out.shape[0]}")
print(f"Output cols : {out.shape[1]}")
print(f"Log entries : {len(log_check)}")
print()
print("Remaining NaN count per country × indicator:")
print("-" * 60)
for cc in TARGET_COUNTRIES:
    sub = out[out["country_code"] == cc]
    for sc in SERIES_CODES:
        nan_ct = int(sub[sc].isna().sum())
        print(f"  {cc} | {sc:<40s} | NaN = {nan_ct}")
    print()


VALIDATION PASSED — all 4 assertions OK
Output rows : 68
Output cols : 14
Log entries : 10

Remaining NaN count per country × indicator:
------------------------------------------------------------
  MYS | AG.LND.PRCP.MM                           | NaN = 1
  MYS | WB_WDI_EN_GHG_ALL_MT_CE_AR5              | NaN = 0
  MYS | WB_WDI_EN_GHG_CO2_PC_CE_AR5              | NaN = 0
  MYS | EN_GHG_CO2_RT_GDP_KD                     | NaN = 0
  MYS | SP.URB.TOTL.IN.ZS                        | NaN = 0
  MYS | EN.CLC.MDAT.ZS                           | NaN = 33
  MYS | AG.LND.FRST.ZS                           | NaN = 0
  MYS | EG.FEC.RNEW.ZS                           | NaN = 2
  MYS | EG.USE.PCAP.KG.OE                        | NaN = 0
  MYS | EG.USE.COMM.FO.ZS                        | NaN = 0
  MYS | NY.GDP.PCAP.CD                           | NaN = 0
  MYS | NV.IND.MANF.ZS                           | NaN = 0

  PHL | AG.LND.PRCP.MM                           | NaN = 1
  PHL | WB_WDI_EN_GHG_ALL_MT_CE_A